# Notebook 5 – Aggregation Features

## Learning Objectives
Understand aggregation features and create count, sum, mean, median, minimum, maximum, standard deviation, group-based, customer-level, product-level, and transaction-level features.

## 1. What are Aggregation Features?
Aggregation features summarize multiple records or values into a single statistic. They help describe customer behaviour, product performance, or transaction patterns.

**Example:** Instead of inspecting every order, a business may calculate each customer's total spending, average order value, or number of purchases.

## 2. Sample E-commerce Dataset
Each row represents a transaction line. We calculate sales amount from quantity and unit price.

In [1]:
import pandas as pd
import numpy as np

transactions = pd.DataFrame({
    "transaction_id": range(1, 11),
    "customer_id": ["C01","C01","C02","C02","C01","C03","C03","C02","C04","C04"],
    "product_id": ["P1","P2","P1","P3","P3","P2","P1","P2","P3","P1"],
    "quantity": [2,1,3,1,2,4,1,2,1,2],
    "unit_price": [100,250,100,400,400,250,100,250,400,100],
    "rating": [5,4,4,3,5,4,5,3,4,5]
})
transactions["sales_amount"] = transactions["quantity"] * transactions["unit_price"]
transactions

,transaction_id,customer_id,product_id,quantity,unit_price,rating,sales_amount
0,1,C01,P1,2,100,5,200
1,2,C01,P2,1,250,4,250
2,3,C02,P1,3,100,4,300
3,4,C02,P3,1,400,3,400
4,5,C01,P3,2,400,5,800
5,6,C03,P2,4,250,4,1000
6,7,C03,P1,1,100,5,100
7,8,C02,P2,2,250,3,500
8,9,C04,P3,1,400,4,400
9,10,C04,P1,2,100,5,200


**Output Explanation:** The dataset contains customer and product identifiers, quantities, prices, ratings, and calculated sales amounts.

## 3. Count Aggregation
Count calculates the number of records in a group.

**Example:** Count transactions per customer.

In [2]:
customer_count = transactions.groupby("customer_id").size().rename("purchase_count").reset_index()
customer_count

,customer_id,purchase_count
0,C01,3
1,C02,3
2,C03,2
3,C04,2


**Output Explanation:** `purchase_count` represents the number of transaction records for each customer.

## 4. Sum Aggregation
Sum adds values within a group.

**Example:** Calculate each customer's total spending.

In [3]:
customer_sum = transactions.groupby("customer_id")["sales_amount"].sum().rename("total_customer_spending").reset_index()
customer_sum

,customer_id,total_customer_spending
0,C01,1250
1,C02,1200
2,C03,1100
3,C04,600


**Output Explanation:** Total customer spending is the sum of transaction sales amounts.

## 5. Mean Aggregation
Mean calculates the arithmetic average.

**Example:** Calculate average transaction value per customer.

In [4]:
customer_mean = transactions.groupby("customer_id")["sales_amount"].mean().rename("average_order_value").reset_index()
customer_mean

,customer_id,average_order_value
0,C01,416.666667
1,C02,400.000000
2,C03,550.000000
3,C04,300.000000


**Output Explanation:** Average order value describes the typical transaction amount for each customer.

## 6. Median Aggregation
Median is the middle value after sorting and is less affected by extreme values than the mean.

**Example:** Calculate median transaction value per customer.

In [5]:
customer_median = transactions.groupby("customer_id")["sales_amount"].median().rename("median_order_value").reset_index()
customer_median

,customer_id,median_order_value
0,C01,250.0
1,C02,400.0
2,C03,550.0
3,C04,300.0


**Output Explanation:** Median spending represents the middle transaction amount for each customer.

## 7. Minimum and Maximum
Minimum and maximum identify the smallest and largest values in a group.

In [6]:
customer_min_max = transactions.groupby("customer_id")["sales_amount"].agg(
    minimum_order_value="min", maximum_order_value="max"
).reset_index()
customer_min_max

,customer_id,minimum_order_value,maximum_order_value
0,C01,200,800
1,C02,300,500
2,C03,100,1000
3,C04,200,400


**Output Explanation:** These features describe the lower and upper transaction-value range for each customer.

## 8. Standard Deviation
Standard deviation measures how spread out values are around their mean.

In [7]:
customer_std = transactions.groupby("customer_id")["sales_amount"].std().fillna(0).rename("spending_std").reset_index()
customer_std

,customer_id,spending_std
0,C01,332.916406
1,C02,100.000000
2,C03,636.396103
3,C04,141.421356


**Output Explanation:** Higher standard deviation indicates more variation in transaction amounts. A single-record group has undefined sample standard deviation; zero is used here for demonstration.

## 9. Group-Based Aggregation
Group-based aggregation applies multiple summaries to groups such as products.

In [8]:
product_aggregation = transactions.groupby("product_id").agg(
    total_quantity=("quantity","sum"),
    total_sales=("sales_amount","sum"),
    average_rating=("rating","mean"),
    transaction_count=("transaction_id","count")
).reset_index()
product_aggregation

,product_id,total_quantity,total_sales,average_rating,transaction_count
0,P1,8,800,4.750000,4
1,P2,7,1750,3.666667,3
2,P3,4,1600,4.000000,3


**Output Explanation:** The table summarizes sales volume, revenue, rating, and transaction count for each product.

## 10. Customer-Level Aggregation
Customer-level aggregation summarizes all eligible transactions for each customer.

In [9]:
customer_features = transactions.groupby("customer_id").agg(
    total_spending=("sales_amount","sum"),
    average_spending=("sales_amount","mean"),
    median_spending=("sales_amount","median"),
    purchase_count=("transaction_id","count"),
    average_quantity=("quantity","mean"),
    average_rating=("rating","mean")
).reset_index()
customer_features

,customer_id,total_spending,average_spending,median_spending,purchase_count,average_quantity,average_rating
0,C01,1250,416.666667,250.0,3,1.666667,4.666667
1,C02,1200,400.000000,400.0,3,2.000000,3.333333
2,C03,1100,550.000000,550.0,2,2.500000,4.500000
3,C04,600,300.000000,300.0,2,1.500000,4.500000


**Output Explanation:** Each row now represents one customer with several customer-behaviour features.

## 11. Product-Level Aggregation
Product-level aggregation summarizes performance for each product.

In [10]:
product_features = transactions.groupby("product_id").agg(
    units_sold=("quantity","sum"),
    product_revenue=("sales_amount","sum"),
    average_product_rating=("rating","mean"),
    number_of_transaction_lines=("transaction_id","count")
).reset_index()
product_features

,product_id,units_sold,product_revenue,average_product_rating,number_of_transaction_lines
0,P1,8,800,4.750000,4
1,P2,7,1750,3.666667,3
2,P3,4,1600,4.000000,3


**Output Explanation:** These features describe product performance and can support product-level analysis.

## 12. Transaction-Level Aggregation
Transaction-level aggregation summarizes line items belonging to the same order. A valid order identifier is required.

In [11]:
order_lines = pd.DataFrame({
    "order_id":[501,501,502,503,503,503],
    "product_id":["P1","P2","P1","P2","P3","P1"],
    "quantity":[1,2,3,1,2,1],
    "unit_price":[100,250,100,250,400,100]
})
order_lines["line_total"] = order_lines["quantity"] * order_lines["unit_price"]
order_features = order_lines.groupby("order_id").agg(
    order_total=("line_total","sum"),
    total_items=("quantity","sum"),
    distinct_products=("product_id","nunique"),
    line_count=("product_id","count")
).reset_index()
order_features

,order_id,order_total,total_items,distinct_products,line_count
0,501,600,3,2,2
1,502,300,3,1,1
2,503,1150,4,3,3


**Output Explanation:** Each row represents an order and describes its value, item quantity, distinct products, and line count.